# Create a New Environment

This tutorial builds a tiny Gymnasium environment, adds MASA labels and costs, registers it with MASA's environment registry, wraps it with `make_env`, and finishes with a small rollout test.

The matching static docs page is at `docs/Tutorials/Environments/Create a New Environment.md`.

## CPU-first setup

Set CPU-friendly defaults before importing MASA modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

## Build the raw Gymnasium environment

`TinyDeliveryEnv` is a deterministic 2x2 grid. The agent starts at `0`, can step through a spill at `1`, has a safe lane at `2`, and receives reward when it reaches the goal at `3`.

In [ ]:
import gymnasium as gym
from gymnasium import spaces


class TinyDeliveryEnv(gym.Env):
    metadata = {"render_modes": ["ansi"]}

    ACTION_NAMES = {
        0: "right",
        1: "down",
        2: "left",
        3: "up",
    }

    def __init__(self):
        self.observation_space = spaces.Discrete(4)
        self.action_space = spaces.Discrete(4)
        self._state = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._state = 0
        return self._state, {"position": self._position()}

    def step(self, action):
        action = int(action)
        assert self.action_space.contains(action), f"Invalid action {action}"

        row, col = self._position()
        if action == 0:  # right
            col = min(col + 1, 1)
        elif action == 1:  # down
            row = min(row + 1, 1)
        elif action == 2:  # left
            col = max(col - 1, 0)
        elif action == 3:  # up
            row = max(row - 1, 0)

        self._state = row * 2 + col
        terminated = self._state == 3
        reward = 1.0 if terminated else 0.0
        return self._state, reward, terminated, False, {"position": (row, col)}

    def render(self):
        symbols = {0: "S", 1: "!", 2: ".", 3: "G"}
        cells = []
        for state in range(4):
            symbol = symbols[state]
            cells.append(f"[{symbol}]" if state == self._state else f" {symbol} ")
        return f"{cells[0]} {cells[1]}\n{cells[2]} {cells[3]}"

    def _position(self):
        return divmod(self._state, 2)

## Check the raw environment

Before adding MASA wrappers, make sure the Gymnasium API works on its own.

In [ ]:
raw_env = TinyDeliveryEnv()
obs, info = raw_env.reset(seed=0)
print(raw_env.render())
print("reset:", obs, info)

obs, reward, terminated, truncated, info = raw_env.step(1)
print("after down:", obs, reward, terminated, truncated, info)
raw_env.close()

## Add labels and costs

MASA constraints read labels from `info["labels"]`. For a CMDP, the cost function maps those labels to a scalar cost.

In [ ]:
def label_fn(obs):
    labels = set()
    if obs == 0:
        labels.add("start")
    if obs == 1:
        labels.add("spill")
    if obs == 3:
        labels.add("goal")
    return labels


def cost_fn(labels):
    return 1.0 if "spill" in labels else 0.0


for state in range(4):
    labels = label_fn(state)
    print(state, sorted(labels), cost_fn(labels))

## Register with MASA

`make_env` resolves environments through MASA's `ENV_REGISTRY`, so this tutorial registers the class there. The guard keeps the cell safe to rerun.

In [ ]:
from masa.common.registry import ENV_REGISTRY
from masa.common.utils import make_env


ENV_ID = "tutorial_tiny_delivery"

if ENV_ID not in ENV_REGISTRY.keys():
    ENV_REGISTRY.register(ENV_ID, TinyDeliveryEnv)


def make_tiny_delivery_env(max_episode_steps=4):
    return make_env(
        ENV_ID,
        "cmdp",
        max_episode_steps,
        label_fn=label_fn,
        cost_fn=cost_fn,
        budget=0.0,
    )

## Inspect the wrapped reset

The base observation is unchanged, while labels and constraint state appear in `info`.

In [ ]:
env = make_tiny_delivery_env()
obs, info = env.reset(seed=0)

print("obs:", obs)
print("labels:", info["labels"])
print("constraint:", info["constraint"])

assert obs == 0
assert info["labels"] == {"start"}
assert info["constraint"]["step"]["cum_cost"] == 0.0
env.close()

## Compare safe, unsafe, and truncated rollouts

The route `[1, 0]` goes down then right through the safe lane. The route `[0, 1]` goes right through the spill before reaching the same goal.

In [ ]:
from IPython.display import Markdown, display


safe_actions = [1, 0]
unsafe_actions = [0, 1]
truncation_actions = [2, 2, 2]


def run_script(actions, *, max_episode_steps=4, seed=0):
    env = make_tiny_delivery_env(max_episode_steps=max_episode_steps)
    obs, info = env.reset(seed=seed)
    rows = []

    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        step_constraint = info["constraint"]["step"]
        episode_constraint = info["constraint"].get("episode", {})
        rows.append(
            {
                "step": step,
                "action": TinyDeliveryEnv.ACTION_NAMES[action],
                "obs": obs,
                "reward": reward,
                "labels": ", ".join(sorted(info["labels"])) or "-",
                "cost": step_constraint["cost"],
                "violation": step_constraint["violation"],
                "cum_cost": step_constraint["cum_cost"],
                "satisfied": episode_constraint.get("satisfied"),
                "terminated": terminated,
                "truncated": truncated,
            }
        )
        if terminated or truncated:
            break

    env.close()
    return rows


def markdown_table(rows):
    columns = ["step", "action", "obs", "reward", "labels", "cost", "violation", "cum_cost", "satisfied", "terminated", "truncated"]
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(str(row[column]) for column in columns) + " |" for row in rows]
    return "\n".join([header, divider, *body])

In [ ]:
safe_rows = run_script(safe_actions)
unsafe_rows = run_script(unsafe_actions)
truncated_rows = run_script(truncation_actions, max_episode_steps=3)

display(Markdown("### Safe route\n" + markdown_table(safe_rows)))
display(Markdown("### Unsafe route\n" + markdown_table(unsafe_rows)))
display(Markdown("### Truncated route\n" + markdown_table(truncated_rows)))

## Write a short rollout test

The same assertions can become a real pytest test once the environment is promoted from notebook example to package code.

In [ ]:
def test_tiny_delivery_wrapped_env():
    env = make_tiny_delivery_env(max_episode_steps=4)
    obs, info = env.reset(seed=0)
    assert obs == 0
    assert info["labels"] == {"start"}
    assert info["constraint"]["step"]["cum_cost"] == 0.0
    env.close()

    safe = run_script(safe_actions)
    assert safe[-1]["obs"] == 3
    assert safe[-1]["reward"] == 1.0
    assert safe[-1]["terminated"] is True
    assert safe[-1]["truncated"] is False
    assert safe[-1]["labels"] == "goal"
    assert safe[-1]["cum_cost"] == 0.0
    assert safe[-1]["satisfied"] == 1.0

    unsafe = run_script(unsafe_actions)
    assert unsafe[0]["obs"] == 1
    assert unsafe[0]["labels"] == "spill"
    assert unsafe[0]["cost"] == 1.0
    assert unsafe[0]["violation"] == 1.0
    assert unsafe[-1]["terminated"] is True
    assert unsafe[-1]["cum_cost"] == 1.0
    assert unsafe[-1]["satisfied"] == 0.0

    truncated = run_script(truncation_actions, max_episode_steps=3)
    assert truncated[-1]["terminated"] is False
    assert truncated[-1]["truncated"] is True
    assert truncated[-1]["cum_cost"] == 0.0


test_tiny_delivery_wrapped_env()
print("TinyDeliveryEnv checks passed")

## What changes when this becomes a package environment?

For a real MASA environment, move the class and helper functions into `masa/envs/...`, add a permanent `ENV_REGISTRY.register(...)` entry in the supported plugins module, and move the rollout assertions into `tests/`. The runtime pattern stays the same: Gymnasium env first, labels and costs second, MASA wrapper stack last.